# Advanced Machine Learning - Final Project
## Anti-Money Laundering (AML) Transaction Classification
**Author:** Giovanni Pacchetti, Asier Larrazabal and Asier Aurre

This notebook addresses the classification of highly imbalanced financial transactions to detect money laundering patterns, applying techniques learned in the Advanced Machine Learning course.

In [10]:
# Install necessary libraries (Run this if using Colab)
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# --- DATA DOWNLOAD OPTIONS ---

# Option 1: Download directly from Kaggle (Recommended)
# You will need your kaggle.json file uploaded to your environment
import os
import shutil
import kagglehub
from pathlib import Path

# Creating a 'data' folder in your current working directory
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# The specific files you requested for your project
FILES_TO_KEEP = [
    "HI-Medium_Patterns.txt",
    "HI-Medium_Trans.csv",
    "HI-Medium_accounts.csv"
]

# --- Download dataset ---
print("Downloading dataset from Kaggle via kagglehub...")
# This will download the dataset to a local cache folder
cache_path = kagglehub.dataset_download("ealtman2019/ibm-transactions-for-anti-money-laundering-aml")
print(f"Dataset downloaded to cache: {cache_path}")

# --- Copy ONLY the required files to your data folder ---
print("\nExtracting and copying specific HI-Medium files...")

# Iterate through the downloaded cache and copy only what we need
for item in Path(cache_path).iterdir():
    # Kaggle sometimes puts files directly there, or inside a subfolder. 
    # This checks if the file matches the ones you want.
    if item.name in FILES_TO_KEEP:
        dest = DATA_DIR / item.name
        shutil.copy(item, dest)
        print(f"✅ Copied: {item.name}")

# --- Verify content ---
print("\nFiles successfully prepared in data/:")
for f in DATA_DIR.iterdir():
    print(" -", f.name)

# Option 2: Download from Google Drive (If you uploaded your preprocessed CSV there)
# !gdown --id "YOUR_FILE_ID_HERE"

Dataset downloaded to cache: C:\Users\asier\.cache\kagglehub\datasets\ealtman2019\ibm-transactions-for-anti-money-laundering-aml\versions\8

Extracting and copying specific HI-Medium files...
✅ Copied: HI-Medium_accounts.csv
✅ Copied: HI-Medium_Patterns.txt
✅ Copied: HI-Medium_Trans.csv

Files successfully prepared in data/:
 - HI-Medium_accounts.csv
 - HI-Medium_Patterns.txt
 - HI-Medium_Trans.csv


In [11]:
import pandas as pd
from pathlib import Path

# Ruta a tu archivo (ajusta según dónde lo guardaste)
DATA_PATH = Path("data/HI-Medium_Trans.csv")  

# Leer dataset
df = pd.read_csv(DATA_PATH)
print("Dataset cargado correctamente!")
print(f"Forma: {df.shape}")
print("\nPrimeras 5 filas:")
df.head()


Dataset cargado correctamente!
Forma: (31898238, 11)

Primeras 5 filas:


,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering
0,2022/09/01 00:17,20,800104D70,20,800104D70,6794.63,US Dollar,6794.63,US Dollar,Reinvestment,0
1,2022/09/01 00:02,3196,800107150,3196,800107150,7739.29,US Dollar,7739.29,US Dollar,Reinvestment,0
2,2022/09/01 00:17,1208,80010E430,1208,80010E430,1880.23,US Dollar,1880.23,US Dollar,Reinvestment,0
3,2022/09/01 00:03,1208,80010E650,20,80010E6F0,73966883.00,US Dollar,73966883.00,US Dollar,Cheque,0
4,2022/09/01 00:02,1208,80010E650,20,80010EA30,45868454.00,US Dollar,45868454.00,US Dollar,Cheque,0


In [12]:
# Leer patterns
from io import StringIO
PATTERNS_PATH = Path("data/HI-Medium_Patterns.txt")
patterns_txt = PATTERNS_PATH.read_text()
lines = [line for line in patterns_txt.split('\n') if ',' in line]
patterns_df = pd.read_csv(StringIO('\n'.join(lines)), header=None)
patterns_df.columns = ['timestamp','id_origen','cuenta_origen','id_destino','cuenta_destino',
                      'monto','moneda_out','monto_in','moneda_in','metodo','flag']

print(f"Patterns: {patterns_df.shape}")
print(patterns_df.head())

Patterns: (22743, 11)
          timestamp  id_origen cuenta_origen  id_destino cuenta_destino  \
0  2022/09/01 05:14        952     8139F54E0      111632      8062C56E0   
1  2022/09/03 13:09     111632     8062C56E0        8456      81363F620   
2  2022/09/01 07:40     118693     823D5EB90       13729      801CF2E60   
3  2022/09/01 14:19      13729     801CF2E60      123621      81A7090F0   
4  2022/09/02 12:40      24750     81363F410      213834      808757B00   

      monto moneda_out  monto_in  moneda_in metodo  flag  
0   5331.44  US Dollar   5331.44  US Dollar    ACH     1  
1   5602.59  US Dollar   5602.59  US Dollar    ACH     1  
2   1400.54  US Dollar   1400.54  US Dollar    ACH     1  
3   1467.94  US Dollar   1467.94  US Dollar    ACH     1  
4  16898.29  US Dollar  16898.29  US Dollar    ACH     1  
